**Objetivo**

Este notebook tiene como objetivo comparar el desempeño de los modelos supervisados entrenados para la clasificación de los niveles de peligro en casos de violencia intrafamiliar y de pareja. Para ello, se analizan las métricas obtenidas en los diferentes escenarios de entrenamiento y se selecciona el modelo con mejor desempeño mediante una evaluación jerárquica de múltiples métricas.

Dado que el propósito principal del estudio es maximizar la identificación de los casos clasificados como Peligro extremo y peligro grave, la selección del modelo prioriza el Recall y el F₂-Score de la clase Peligro extremo. Como criterios complementarios se consideran el F1-Score Macro y el Accuracy, buscando un equilibrio entre la capacidad de detección de la clase crítica y el desempeño general del clasificador.

Finalmente, se realiza un análisis detallado del modelo seleccionado mediante matrices de confusión y métricas por clase, con énfasis en las clases Peligro extremo y Peligro grave, por ser las categorías de mayor interés para la priorización del riesgo.

In [0]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from pyspark.sql import functions as F

from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

import mlflow
from mlflow.tracking import MlflowClient

**Carga de los resultados de los modelos**

In [0]:
# Cargar resultados de los modelos

df_logistica = spark.table(
    "ml_proyecto_7405607705157039.default.metricas_regresion_logistica"
)

df_rf = spark.table(
    "ml_proyecto_7405607705157039.default.metricas_random_forest"
)

df_svm = spark.table(
    "ml_proyecto_7405607705157039.default.metricas_svm"
)

In [0]:
# Unir resultados

df_resultados = (
    df_logistica
    .unionByName(df_rf)
    .unionByName(df_svm)
)

In [0]:
#verificar esquema de los resultados
df_resultados.printSchema()

In [0]:
#visualizar los resultados
display(df_resultados)

**Comparación descriptiva del desempeño de los modelos**

In [0]:
#convertir a pandas la tabla de resultados

pdf_resultados = df_resultados.toPandas()

In [0]:
#Ordenar resultados
pdf_resultados = pdf_resultados.sort_values(
    by=["modelo", "escenario"]
).reset_index(drop=True)

pdf_resultados

In [0]:
#Verificar dimensiones de la tabla de resualtados

print(f"Número de modelos evaluados: {pdf_resultados['modelo'].nunique()}")

print(f"Número de escenarios: {pdf_resultados['escenario'].nunique()}")

print(f"Total de evaluaciones: {len(pdf_resultados)}")

In [0]:
#Tabla resumen para el análisis de resultados

columnas = [
    "modelo",
    "escenario",
    "accuracy",
    "recall_extremo",
    "f2_extremo",
    "f1_macro"
]

pdf_resultados[columnas]

**Selección jerárquica del mejor modelo**

La selección del modelo se realizó mediante una evaluación jerárquica de múltiples métricas de desempeño. En primer lugar, se priorizó el Recall de la clase Peligro extremo, dado que el objetivo principal del estudio es maximizar la identificación de los casos de mayor riesgo. Posteriormente, se consideró el F₂-Score de esta clase, por otorgar mayor importancia al Recall sin descuidar completamente la Precisión. Como criterios complementarios se analizaron el F1-Score y el Accuracy macro o global, con el propósito de seleccionar un modelo con un desempeño equilibrado tanto en la clase crítica como en el conjunto de clases.

Medidas de desempeño de clasificación

VP = verdaderos positivos

FP = Falsos positivos

VN = verdaderos negativos

FN = Falsos negativos

N = Obsrvaciones totales

- Recall (sensibilidad): mide que también clasificó las observaciones de la clase. VP/(VP+FN)

En el caso de la clase peligro extremo, cuantas veces acertó en la clasificación de las observaciones de esta clase.

-Precisión: Porcentaje de observaciones que fueron clasificadas correctamente en la clase positiva, del total de observaciones clasificadas como positivas. VP/(VP+FP)

En el caso de la clase de peligro extremo, cuantas veces acertó en la clasificación de esta clase, dentro del total observaciones clasificadas como peligro extremo.

- F₂-score (F Beta-score, Beta > 2):  F beta-Score, es la media armónica ponderada entre precisión y recall. 
                                      (1+ Beta-score al cuadrado) * precisión*recall/Beta-score al cuadrado *(precisión*recall)
                                      Esta medida permite asignar diferente importancia a la precisión y al recall.

                                      Para un Beta > 2, se le esta asignando 4 veces más importancia al recall que a la precisión. Por lo tanto, se castigando la precisión para la selección de un modelo capaz de identificar la mayor cantidad posible de casos críticos o de peligro extremo. lo que resulta apropiado cuando el costo de no identificar un caso crítico es superior al de generar una alerta adicional.

                

-F1-score: es el promedio armónico entre la precisión y la sensibilidad. F1= 2 precision*recall/precision+recall.

El F1-score busca un equilibrio entre la precisión y el recall. Si la precisión es muy baja, F1 baja. Si el recall es muy bajo, F1 también baja. Solo es alto cuando ambos son altos.

-Accuracy (exactitud): mide la capacidad que tiene el modelo de predecir los valores reales según su clase. VP + VN /N

Para la evaluación de los modelos será una medida global es decir, establecerá el prcentaje de observaciones bien clasificada según su clase.

















                                      









In [0]:
#Ordenar por prioridad

ranking_modelos = (
    pdf_resultados
    .sort_values(
        by=[
            "recall_extremo",
            "f2_extremo",
            "f1_macro",
            "accuracy"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

ranking_modelos

In [0]:
#Agregar posición

ranking_modelos.insert(
    0,
    "ranking",
    range(1, len(ranking_modelos) + 1)
)

ranking_modelos

In [0]:
# Mejor modelo según la selección jerárquica

columnas_seleccion = [
    "modelo",
    "escenario",
    "recall_extremo",
    "f2_extremo",
    "f1_macro",
    "accuracy"
]

mejor_modelo = ranking_modelos.loc[
    [0],
    columnas_seleccion
]

mejor_modelo

In [0]:
mejor_modelo = (
    ranking_modelos.loc[
        [0],
        columnas_seleccion
    ]
    .rename(columns={
        "modelo": "Modelo",
        "escenario": "Escenario",
        "recall_extremo": "Recall (Peligro extremo)",
        "f2_extremo": "F₂-Score (Peligro extremo)",
        "f1_macro": "F1 Macro",
        "accuracy": "Accuracy"
    })
    .round(4)
)

mejor_modelo

In [0]:
# Obtener información del modelo ganador

modelo_ganador_info = ranking_modelos.iloc[0]

run_id_ganador = modelo_ganador_info["run_id"]

print(f"Run ID ganador : {run_id_ganador}")
print(f"Modelo         : {modelo_ganador_info['modelo']}")
print(f"Escenario      : {modelo_ganador_info['escenario']}")

In [0]:
# Construir URI del modelo ganador

modelo_uri = f"runs:/{run_id_ganador}/modelo"

print("URI del modelo:")
print(modelo_uri)

In [0]:
# Cargar modelo ganador

import mlflow

modelo_ganador = mlflow.sklearn.load_model(modelo_uri)

print("Modelo cargado correctamente.")
print(type(modelo_ganador))

**Generación de predicciones del modelo seleccionado**

In [0]:
#Revisar estructura del modelo cargado

# Estructura del modelo cargado

modelo_ganador

**Generación de predicciones**

In [0]:
#Cargar el conjunto de prueba

# Cargar conjunto de prueba

df_test = spark.table(
    "ml_proyecto_7405607705157039.default.test"
)

df_test.printSchema()

In [0]:
# Convertir a Pandas

pdf_test = df_test.toPandas()

In [0]:
# Variables del modelo

variables_modelo = [
    "sexo_victima_cod",
    "ciclo_vital_cod",
    "escolaridad_cod",
    "estado_civil_cod",
    "contexto_del_hecho_cod",
    "mecanismo_causal_cod",
    "presunto_agresor_cod"
]

In [0]:
#Construir X_test y y_test

# Separar variables predictoras y objetivo

X_test = pdf_test[variables_modelo]

y_test = pdf_test["nivel_peligro"]

In [0]:
#Verificaciones
print("Dimensiones X_test:", X_test.shape)
print("Dimensiones y_test:", y_test.shape)

print("\nColumnas de X_test:")
print(X_test.columns.tolist())

**Generar las predicciones**

In [0]:
# Generar predicciones del modelo ganador

y_pred = modelo_ganador.predict(X_test)

In [0]:
#verificar predicciones
print("Observaciones reales :", len(y_test))
print("Predicciones         :", len(y_pred))

In [0]:
#Distribución de las clases

tabla_clases = pd.DataFrame({
    "Real": y_test.value_counts(),
    "Predicha": pd.Series(y_pred).value_counts()
}).fillna(0).astype(int)

tabla_clases

**Matriz de confusión multiclase**

In [0]:
#Calcular la matriz de confusión

clases = [
    "Peligro bajo",
    "Peligro moderado",
    "Peligro grave",
    "Peligro extremo"
]

matriz_confusion = confusion_matrix(
    y_test,
    y_pred,
    labels=clases
)

In [0]:
#Mostrar la matriz como tabla

df_matriz = pd.DataFrame(
    matriz_confusion,
    index=clases,
    columns=clases
)

df_matriz

In [0]:

matriz_normalizada = (
    matriz_confusion.astype(float)
    / matriz_confusion.sum(axis=1)[:, np.newaxis]
)

df_matriz_normalizada = pd.DataFrame(
    matriz_normalizada,
    index=clases,
    columns=clases
).round(3)

df_matriz_normalizada


In [0]:
#Visualización gráfica de la matriz de confusión

# Matriz de confusión (valores absolutos)

fig, ax = plt.subplots(figsize=(8, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz_confusion,
    display_labels=clases
)

disp.plot(
    ax=ax,
    values_format="d",
    colorbar=False
)

plt.title("Matriz de confusión multiclase")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [0]:
#Matriz de confusión normalizada

# Convertir a porcentaje
matriz_normalizada_pct = matriz_normalizada * 100

fig, ax = plt.subplots(figsize=(8, 7))

disp = ConfusionMatrixDisplay(
    confusion_matrix=matriz_normalizada_pct,
    display_labels=clases
)

disp.plot(
    ax=ax,
    values_format=".1f",   # Una cifra decimal
    colorbar=False
)

# Agregar el símbolo % a cada celda
for texto in disp.text_.ravel():
    texto.set_text(texto.get_text() + "%")

plt.title("Matriz de confusión multiclase normalizada")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [0]:
# Función de análisis por clase

from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score
)

import pandas as pd


def analizar_clase(y_real, y_pred, clase):

    # Clasificación binaria
   
    y_real_bin = (y_real == clase)
    y_pred_bin = (y_pred == clase)

    tn, fp, fn, tp = confusion_matrix(
        y_real_bin,
        y_pred_bin
    ).ravel()

    # Tabla VP FP FN VN
   
    tabla_vp = pd.DataFrame({

        "Métrica":[
            "Verdaderos Positivos (VP)",
            "Falsos Positivos (FP)",
            "Falsos Negativos (FN)",
            "Verdaderos Negativos (VN)"
        ],

        "Valor":[
            tp,
            fp,
            fn,
            tn
        ]

    })

    # Matriz binaria
    
    matriz_binaria = pd.DataFrame(

        [
            [tp, fn],
            [fp, tn]
        ],

        index=[
            f"Real: {clase}",
            f"Real: No {clase}"
        ],

        columns=[
            f"Predicho: {clase}",
            f"Predicho: No {clase}"
        ]

    )

    # Métricas
    
    precision = precision_score(
        y_real_bin,
        y_pred_bin,
        zero_division=0
    )

    recall = recall_score(
        y_real_bin,
        y_pred_bin,
        zero_division=0
    )

    f1 = f1_score(
        y_real_bin,
        y_pred_bin,
        zero_division=0
    )

    f2 = fbeta_score(
        y_real_bin,
        y_pred_bin,
        beta=2,
        zero_division=0
    )

    specificity = tn / (tn + fp)

    metricas = pd.DataFrame({

        "Métrica":[
            "Precisión",
            "Recall",
            "F1-Score",
            "F₂-Score",
            "Especificidad"
        ],

        "Valor":[
            precision,
            recall,
            f1,
            f2,
            specificity
        ]

    })

    metricas["Valor"] = (
        metricas["Valor"] * 100
    ).map(lambda x: f"{x:.2f}%")

    return tabla_vp, matriz_binaria, metricas

In [0]:
# Análisis de la clase Peligro extremo

tabla_extremo, matriz_extremo, metricas_extremo = analizar_clase(
    y_test,
    y_pred,
    "Peligro extremo"
)

print("========== PELIGRO EXTREMO ==========")

display(tabla_extremo)

display(matriz_extremo)

display(metricas_extremo)

In [0]:
# Análisis de la clase Peligro grave
tabla_grave, matriz_grave, metricas_grave = analizar_clase(
    y_test,
    y_pred,
    "Peligro grave"
)

print("========== PELIGRO GRAVE ==========")

display(tabla_grave)

display(matriz_grave)

display(metricas_grave)